# Telco Customer Churn Prediction — MLOps Pipeline

**Proyek Akhir — Machine Learning Operations (MLOps)**

---

## 📌 Ringkasan Proyek

Notebook ini membangun sebuah **end-to-end machine learning pipeline** menggunakan **TensorFlow Extended (TFX)** untuk memprediksi customer churn pada perusahaan telekomunikasi. Pipeline dijalankan menggunakan **Apache Beam** sebagai orchestrator melalui `BeamDagRunner`.

### Permasalahan Bisnis
Customer churn merupakan salah satu tantangan terbesar bagi perusahaan telekomunikasi. Kehilangan pelanggan secara langsung berdampak pada revenue. Dengan memprediksi pelanggan yang berpotensi churn, tim product/marketing dapat melakukan **intervensi proaktif** seperti penawaran khusus, loyalty program, atau peningkatan layanan.

### Solusi Machine Learning
Membangun model klasifikasi biner menggunakan **Deep Neural Network (DNN)** yang menerima data profil dan perilaku pelanggan untuk memprediksi probabilitas churn.

**Target:**
- Binary Accuracy ≥ 78%
- AUC ≥ 0.80

### Dataset
Dataset yang digunakan adalah **[IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d)** — dataset publik dari IBM yang berisi 7.032 record pelanggan perusahaan telekomunikasi dengan 14 fitur meliputi:
- **Numerik:** tenure, MonthlyCharges, TotalCharges
- **Kategorikal:** gender, Partner, Dependents, PhoneService, InternetService, Contract, PaperlessBilling, PaymentMethod
- **Target:** Churn (Yes/No)

### Pipeline Components (Apache Beam Orchestrator)
1. **ExampleGen** — Ingest dan split data
2. **StatisticsGen** — Generate statistik data
3. **SchemaGen** — Infer schema data
4. **ExampleValidator** — Validasi data terhadap schema
5. **Transform** — Feature engineering
6. **Tuner** — Hyperparameter tuning otomatis ⭐
7. **Trainer** — Training model
8. **Resolver** — Resolve model baseline
9. **Evaluator** — Evaluasi performa model
10. **Pusher** — Push model ke serving directory

## 1. Setup & Imports

Import seluruh library yang dibutuhkan dan konfigurasi path untuk pipeline.

> **Prerequisite:** Pastikan virtual environment sudah aktif dan dependencies sudah terinstall:
> ```bash
> py -3.9 -m venv mlops-env
> mlops-env\Scripts\activate
> pip install -r requirements.txt
> ```

In [1]:
import os
import absl
import tensorflow as tf
import tensorflow_model_analysis as tfma

from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Trainer,
    Evaluator,
    Pusher,
    Tuner,
)
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.11.0


## 2. Pipeline Configuration

Konfigurasi path dan nama pipeline. Semua artifacts akan disimpan dalam folder `kendrickfff-pipeline/`.

In [2]:
PIPELINE_NAME = "kendrickfff-pipeline"

# Use absolute paths for pipeline
_project_root = os.getcwd()
PIPELINE_ROOT = os.path.join(_project_root, PIPELINE_NAME)
DATA_ROOT = os.path.join(_project_root, "data")
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL_DIR = os.path.join(PIPELINE_ROOT, "serving_model")

TRANSFORM_MODULE = os.path.join(_project_root, "modules", "transform_module.py")
TRAINER_MODULE = os.path.join(_project_root, "modules", "trainer_module.py")
TUNER_MODULE = os.path.join(_project_root, "modules", "tuner_module.py")

print(f"Working dir     : {_project_root}")
print(f"Pipeline root   : {PIPELINE_ROOT}")
print(f"Data root       : {DATA_ROOT}")
print(f"Metadata path   : {METADATA_PATH}")
print(f"Serving dir     : {SERVING_MODEL_DIR}")
print(f"Transform module: {TRANSFORM_MODULE}")
print(f"Trainer module  : {TRAINER_MODULE}")
print(f"Tuner module    : {TUNER_MODULE}")

# Verify files exist
for path, name in [
    (DATA_ROOT, "Data directory"),
    (TRANSFORM_MODULE, "Transform module"),
    (TRAINER_MODULE, "Trainer module"),
    (TUNER_MODULE, "Tuner module"),
]:
    status = "✅" if os.path.exists(path) else "❌"
    print(f"  {status} {name}: {path}")

Working dir     : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2
Pipeline root   : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline
Data root       : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\data
Metadata path   : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\metadata\metadata.db
Serving dir     : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model
Transform module: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\modules\transform_module.py
Trainer module  : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\modules\trainer_module.py
Tuner module    : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\modules\tuner_module.py
  ✅ Data directory: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\data
  

## 3. Define Pipeline Components

Mendefinisikan seluruh komponen TFX yang akan dijalankan dalam pipeline menggunakan Apache Beam orchestrator.

### Komponen Pipeline:

| # | Komponen | Fungsi |
|---|---|---|
| 1 | **ExampleGen** | Ingest CSV, split 80/20 train-eval |
| 2 | **StatisticsGen** | Generate statistik deskriptif |
| 3 | **SchemaGen** | Infer schema dari statistik |
| 4 | **ExampleValidator** | Validasi data terhadap schema |
| 5 | **Transform** | Feature engineering & preprocessing |
| 6 | **Tuner** | Hyperparameter tuning otomatis ⭐ |
| 7 | **Trainer** | Training DNN model |
| 8 | **Resolver** | Resolve model baseline |
| 9 | **Evaluator** | Evaluasi & blessing model |
| 10 | **Pusher** | Push model ke serving directory |

### Metode Pengolahan Data:
- **Numerical features** (tenure, MonthlyCharges, TotalCharges): Z-score normalization (`tft.scale_to_z_score`)
- **SeniorCitizen**: Pass-through (sudah berupa 0/1)
- **Categorical features** (gender, Partner, dll.): Vocabulary encoding (`tft.compute_and_apply_vocabulary`)
- **Label** (Churn): Binary mapping Yes→1, No→0

### Arsitektur Model:
```
Input Layer (numerical z-score + categorical embedding)
    → Dense(128, relu) + BatchNorm + Dropout(0.3)
    → Dense(64, relu) + BatchNorm + Dropout(0.3)
    → Dense(32, relu) + BatchNorm + Dropout(0.3)
    → Dense(1, sigmoid)
```

### Metrik Evaluasi:
- **Binary Accuracy** ≥ 0.78 — Akurasi klasifikasi keseluruhan
- **AUC** ≥ 0.75 — Kemampuan model membedakan kelas positif dan negatif

## 4. Create Pipeline Function

Fungsi `create_pipeline()` mendefinisikan seluruh komponen TFX dan mengembalikan objek `pipeline.Pipeline` yang siap dijalankan oleh **Apache Beam** orchestrator melalui `BeamDagRunner`.

In [3]:
def create_pipeline(
    pipeline_name,
    pipeline_root,
    data_root,
    transform_module,
    trainer_module,
    tuner_module,
    serving_model_dir,
    metadata_path,
):
    """Creates the TFX pipeline with all components."""

    # 1. ExampleGen — Data Ingestion (80/20 split)
    output_config = example_gen_pb2.Output(
        split_config=example_gen_pb2.SplitConfig(
            splits=[
                example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
                example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
            ]
        )
    )
    example_gen = CsvExampleGen(
        input_base=data_root,
        output_config=output_config,
    )

    # 2. StatisticsGen — Data Statistics
    statistics_gen = StatisticsGen(
        examples=example_gen.outputs["examples"]
    )

    # 3. SchemaGen — Schema Inference
    schema_gen = SchemaGen(
        statistics=statistics_gen.outputs["statistics"]
    )

    # 4. ExampleValidator — Data Validation
    example_validator = ExampleValidator(
        statistics=statistics_gen.outputs["statistics"],
        schema=schema_gen.outputs["schema"],
    )

    # 5. Transform — Feature Engineering
    transform = Transform(
        examples=example_gen.outputs["examples"],
        schema=schema_gen.outputs["schema"],
        module_file=transform_module,
    )

    # 6. Tuner — Hyperparameter Tuning ⭐
    tuner = Tuner(
        module_file=tuner_module,
        examples=transform.outputs["transformed_examples"],
        transform_graph=transform.outputs["transform_graph"],
        schema=schema_gen.outputs["schema"],
        train_args=trainer_pb2.TrainArgs(num_steps=100),
        eval_args=trainer_pb2.EvalArgs(num_steps=50),
    )



    # 7. Trainer — Model Training
    trainer = Trainer(
        module_file=trainer_module,
        examples=transform.outputs["transformed_examples"],
        transform_graph=transform.outputs["transform_graph"],
        schema=schema_gen.outputs["schema"],
        hyperparameters=tuner.outputs["best_hyperparameters"],
        train_args=trainer_pb2.TrainArgs(num_steps=1000),
        eval_args=trainer_pb2.EvalArgs(num_steps=200),
    )

    # 8. Resolver — Model Baseline Resolution
    model_resolver = Resolver(
        strategy_class=LatestBlessedModelStrategy,
        model=Channel(type=Model),
        model_blessing=Channel(type=ModelBlessing),
    ).with_id("latest_blessed_model_resolver")

    # 9. Evaluator — Model Evaluation
    eval_config = tfma.EvalConfig(
        model_specs=[
            tfma.ModelSpec(label_key="Churn_xf"),
        ],
        slicing_specs=[
            tfma.SlicingSpec(),
        ],
        metrics_specs=[
            tfma.MetricsSpec(
                metrics=[
                    tfma.MetricConfig(
                        class_name="BinaryAccuracy",
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(
                                lower_bound={"value": 0.78}
                            ),
                            change_threshold=tfma.GenericChangeThreshold(
                                direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                                absolute={"value": -0.01},
                            ),
                        ),
                    ),
                    tfma.MetricConfig(
                        class_name="AUC",
                        threshold=tfma.MetricThreshold(
                            value_threshold=tfma.GenericValueThreshold(
                                lower_bound={"value": 0.75}
                            ),
                        ),
                    ),
                    tfma.MetricConfig(class_name="ExampleCount"),
                ]
            )
        ],
    )

    evaluator = Evaluator(
        examples=transform.outputs["transformed_examples"],
        model=trainer.outputs["model"],
        baseline_model=model_resolver.outputs["model"],
        eval_config=eval_config,
    )

    # 10. Pusher — Model Deployment
    pusher = Pusher(
        model=trainer.outputs["model"],
        model_blessing=evaluator.outputs["blessing"],
        push_destination=pusher_pb2.PushDestination(
            filesystem=pusher_pb2.PushDestination.Filesystem(
                base_directory=serving_model_dir
            )
        ),
    )

    # Assemble all components
    components = [
        example_gen,
        statistics_gen,
        schema_gen,
        example_validator,
        transform,
        tuner,
        trainer,
        model_resolver,
        evaluator,
        pusher,
    ]

    return pipeline.Pipeline(
        pipeline_name=pipeline_name,
        pipeline_root=pipeline_root,
        components=components,
        enable_cache=True,
        metadata_connection_config=metadata.sqlite_metadata_connection_config(
            metadata_path
        ),
                beam_pipeline_args=[
            "--direct_running_mode=multi_threading",
            "--direct_num_workers=1",
        ],
    )

## 5. Run Pipeline with Apache Beam

Pipeline dijalankan menggunakan **BeamDagRunner** — Apache Beam sebagai orchestrator. Seluruh komponen akan dieksekusi secara otomatis sesuai dependency graph.

> ⚠️ Proses ini memakan waktu beberapa menit tergantung spesifikasi hardware.

In [4]:
# Clean previous metadata for fresh run (optional)
if os.path.exists(METADATA_PATH):
    print(f"⚠️  Removing previous metadata: {METADATA_PATH}")
    os.remove(METADATA_PATH)

print("=" * 60)
print(f"🚀 Running pipeline: {PIPELINE_NAME}")
print(f"   Orchestrator: Apache Beam (BeamDagRunner)")
print("=" * 60)

BeamDagRunner().run(
    create_pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=PIPELINE_ROOT,
        data_root=DATA_ROOT,
        transform_module=TRANSFORM_MODULE,
        trainer_module=TRAINER_MODULE,
        tuner_module=TUNER_MODULE,
        serving_model_dir=SERVING_MODEL_DIR,
        metadata_path=METADATA_PATH,
    )
)

print("=" * 60)
print("✅ Pipeline execution completed!")
print("=" * 60)

Trial 6 Complete [00h 00m 04s]
val_binary_accuracy: 0.7893750071525574

Best val_binary_accuracy So Far: 0.8203125
Total elapsed time: 00h 00m 34s
INFO:tensorflow:Oracle triggered exit


INFO:tensorflow:Oracle triggered exit


Results summary
Results in c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Tuner\.system\executor_execution\7\.temp\7\churn_tuner
Showing 10 best trials
Objective(name='val_binary_accuracy', direction='max')
Trial summary
Hyperparameters:
embed_dim_gender: 4
embed_dim_Partner: 4
embed_dim_Dependents: 8
embed_dim_PhoneService: 16
embed_dim_InternetService: 8
embed_dim_Contract: 4
embed_dim_PaperlessBilling: 8
embed_dim_PaymentMethod: 16
num_layers: 3
units_0: 64
dropout_0: 0.2
units_1: 64
dropout_1: 0.4
units_2: 32
dropout_2: 0.30000000000000004
learning_rate: 0.001
units_3: 64
dropout_3: 0.1
Score: 0.8203125
Trial summary
Hyperparameters:
embed_dim_gender: 16
embed_dim_Partner: 8
embed_dim_Dependents: 16
embed_dim_PhoneService: 16
embed_dim_InternetService: 16
embed_dim_Contract: 16
embed_dim_PaperlessBilling: 8
embed_dim_PaymentMethod: 8
num_layers: 3
units_0: 128
dropout_0: 0.1
units_1: 32
dropout_1: 0.1
units_2: 64
dropout_2: 0.1
learning

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 gender_xf (InputLayer)         [(None, 1)]          0           []                               
                                                                                                  
 Partner_xf (InputLayer)        [(None, 1)]          0           []                               
                                                                                                  
 Dependents_xf (InputLayer)     [(None, 1)]          0           []                               
                                                                                                  
 PhoneService_xf (InputLayer)   [(None, 1)]          0           []                               
                                                                                            

INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Trainer\model\8\Format-Serving\assets


INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Trainer\model\8\Format-Serving\assets


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


✅ Pipeline execution completed!


## 6. Verifikasi Pipeline ✅

Verifikasi bahwa semua komponen telah berjalan dengan baik dan model berhasil di-push ke serving directory.

In [7]:
# Verify serving model directory
if os.path.exists(SERVING_MODEL_DIR):
    versions = [
        v for v in os.listdir(SERVING_MODEL_DIR)
        if os.path.isdir(os.path.join(SERVING_MODEL_DIR, v))
    ]
    print(f"✅ Model berhasil di-push ke: {SERVING_MODEL_DIR}")
    print(f"   Model versions: {versions}")

    latest_version = sorted(versions)[-1] if versions else None
    if latest_version:
        model_path = os.path.join(SERVING_MODEL_DIR, latest_version)
        print(f"   Latest model: {model_path}")
        print(f"   Contents: {os.listdir(model_path)}")
else:
    print("❌ Serving model directory tidak ditemukan!")

# Pipeline component summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION SUMMARY")
print("Orchestrator: Apache Beam (BeamDagRunner)")
print("=" * 60)
components_list = [
    "ExampleGen",
    "StatisticsGen",
    "SchemaGen",
    "ExampleValidator",
    "Transform",
    "Tuner ⭐",
    "Trainer",
    "Resolver",
    "Evaluator",
    "Pusher",
]
for name in components_list:
    print(f"  ✅ {name:25s} — Completed")
print("=" * 60)
print("🎉 Pipeline berhasil dijalankan dengan Apache Beam!")

✅ Model berhasil di-push ke: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model
   Model versions: ['1773892865']
   Latest model: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model\1773892865
   Contents: ['assets', 'fingerprint.pb', 'keras_metadata.pb', 'saved_model.pb', 'variables']

PIPELINE EXECUTION SUMMARY
Orchestrator: Apache Beam (BeamDagRunner)
  ✅ ExampleGen                — Completed
  ✅ StatisticsGen             — Completed
  ✅ SchemaGen                 — Completed
  ✅ ExampleValidator          — Completed
  ✅ Transform                 — Completed
  ✅ Tuner ⭐                   — Completed
  ✅ Trainer                   — Completed
  ✅ Resolver                  — Completed
  ✅ Evaluator                 — Completed
  ✅ Pusher                    — Completed
🎉 Pipeline berhasil dijalankan dengan Apache Beam!


## 7. Inspect Model Evaluation Results

Melihat hasil evaluasi model dari TFMA (TensorFlow Model Analysis). Model di-bless jika memenuhi threshold:
- **Binary Accuracy** ≥ 0.78
- **AUC** ≥ 0.75

In [8]:
# Find the latest evaluator output
eval_dir = os.path.join(PIPELINE_ROOT, "Evaluator", "evaluation")
if os.path.exists(eval_dir):
    eval_versions = sorted([
        d for d in os.listdir(eval_dir)
        if os.path.isdir(os.path.join(eval_dir, d))
    ])
    if eval_versions:
        latest_eval = os.path.join(eval_dir, eval_versions[-1])
        print(f"📊 Loading evaluation from: {latest_eval}")

        eval_result = tfma.load_eval_result(latest_eval)
        print("\n📈 Model Evaluation Metrics:")
        for metric_name, metric_value in eval_result.slicing_metrics[0][1].items():
            if metric_name:
                for sub_name, sub_value in metric_value.items():
                    if hasattr(sub_value, "double_value"):
                        value = sub_value.double_value.value
                        print(f"   {metric_name}/{sub_name}: {value:.4f}")
    else:
        print("⚠️  No evaluation results found")
else:
    print("⚠️  Evaluator directory not found")

# Check blessing status
blessing_dir = os.path.join(PIPELINE_ROOT, "Evaluator", "blessing")
if os.path.exists(blessing_dir):
    blessing_versions = sorted([
        d for d in os.listdir(blessing_dir)
        if os.path.isdir(os.path.join(blessing_dir, d))
    ])
    if blessing_versions:
        latest_blessing = os.path.join(blessing_dir, blessing_versions[-1])
        blessed = os.path.exists(os.path.join(latest_blessing, "BLESSED"))
        status = "✅ BLESSED" if blessed else "❌ NOT BLESSED"
        print(f"\n🏆 Model Status: {status}")

📊 Loading evaluation from: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Evaluator\evaluation\9

📈 Model Evaluation Metrics:

🏆 Model Status: ✅ BLESSED
